In [1]:
import json
import pandas as pd

# Load true IPD
df_true = pd.read_csv("../data/processed/true_ipd_trial_001.csv")

# Load KM-style observed inputs
curve_points = pd.read_csv("../data/raw/km_curve_points_trial_001.csv")
censor_points = pd.read_csv("../data/raw/km_censor_points_trial_001.csv")

# Load reconstruction-ready inputs
with open("../data/raw/km_inputs_trial_001.json", "r") as f:
    inputs = json.load(f)

n = inputs["n"]
t = inputs["t"]
S = inputs["S"]
cens_t = inputs["cens_t"]

print("Loaded true IPD shape:", df_true.shape)
print("Loaded curve points shape:", curve_points.shape)
print("Loaded censor points shape:", censor_points.shape)
print("Initial cohort size:", n)
print("Number of KM steps:", len(t))
print("Number of censor points:", len(cens_t))

Loaded true IPD shape: (300, 7)
Loaded curve points shape: (256, 2)
Loaded censor points shape: (22, 3)
Initial cohort size: 300
Number of KM steps: 256
Number of censor points: 22


# IPD Reconstruction

This notebook reconstructs individual patient data (IPD) from Kaplan–Meier (KM) representations.

We compare two reconstruction pipelines:

1. our reconstruction implementation  
2. the original RESOLVE-IPD implementation  

Both methods use as input:
- KM step times
- KM survival probabilities
- censoring times (when available)

The output is a reconstructed dataset with:
- observed time
- event indicator

## Observable Inputs for IPD Reconstruction

In practice, individual patient data (IPD) are not directly available. Instead, published survival analyses typically provide only aggregate information derived from Kaplan–Meier (KM) curves.

The reconstruction algorithm operates on the following observable quantities:

- **KM step times ($t$)**: the time points at which the survival curve decreases  
- **Survival probabilities ($S$)**: the estimated survival levels immediately after each step  
- **Censoring times ($cens\_t$)** *(when available)*: approximate locations of censoring marks on the KM curve  
- **Initial cohort size ($n$)**: the number of individuals at risk at time zero  

These inputs collectively define the observed survival process. The goal of reconstruction is to infer a plausible set of individual-level event times and censoring indicators such that the resulting KM curve matches the observed $(t, S)$ as closely as possible.

Importantly, this is an **inverse problem**: multiple IPD datasets may be consistent with the same observed KM curve. Reconstruction methods therefore rely on additional assumptions or constraints (e.g., censoring patterns, risk tables) to obtain a reasonable solution.

In [2]:
t = curve_points["time"].tolist()
S = curve_points["survival"].tolist()
cens_t = censor_points["time"].tolist()

n = len(df_true)

print("Initial cohort size:", n)
print("Number of step points:", len(t))
print("Number of censor points:", len(cens_t))

Initial cohort size: 300
Number of step points: 256
Number of censor points: 22


In [3]:
# Our Method
from src.cen_km.reconstruct import reconstruct_ipd_with_censoring

ours = reconstruct_ipd_with_censoring(
    curve_points=curve_points,
    censor_points=censor_points,
    n_initial=n,
)

ipd_ours = ours.ipd
event_table_ours = ours.event_table

ipd_ours.head(), event_table_ours.head()

(       time  event
 0  0.042745      1
 1  0.100326      1
 2  0.154774      1
 3  0.220988      1
 4  0.331674      0,
        time  survival_target  survival_reconstructed  n_risk  n_events  \
 0  0.000000         1.000000                1.000000     300         0   
 1  0.042745         0.996667                0.996667     300         1   
 2  0.100326         0.993333                0.993333     299         1   
 3  0.154774         0.990000                0.990000     298         1   
 4  0.220988         0.986667                0.986667     297         1   
 
    n_censored  interval_error  
 0           0             NaN  
 1           0    4.440892e-16  
 2           0    2.220446e-16  
 3           0    4.440892e-16  
 4           0    1.110223e-16  )

In [4]:
# RESOLVE-IPD
from src.external_bridge.resolve_ipd_adapter import reconstruct_ipd_with_resolve

ipd_resolve = reconstruct_ipd_with_resolve(
    n=n,
    t=t,
    S=S,
    cens_t=cens_t,
    random_state=733,
    debug=False,
)

ipd_resolve.head()

,time,event
0,0.042745,1
1,0.100326,1
2,0.154774,1
3,0.220988,1
4,0.331674,0


In [5]:
# basic reconstruction summaries
print("True IPD shape:", df_true.shape)
print("Our reconstruction shape:", ipd_ours.shape)
print("RESOLVE-IPD shape:", ipd_resolve.shape)

print("\nEvent counts")
print("True:", df_true["event"].sum())
print("Ours:", ipd_ours["event"].sum())
print("RESOLVE-IPD:", ipd_resolve["event"].sum())

True IPD shape: (300, 7)
Our reconstruction shape: (300, 2)
RESOLVE-IPD shape: (300, 2)

Event counts
True: 233
Ours: 233
RESOLVE-IPD: 233


In [6]:
event_table_ours.head(10)

,time,survival_target,survival_reconstructed,n_risk,n_events,n_censored,interval_error
0,0.000000,1.000000,1.000000,300,0,0,NaN
1,0.042745,0.996667,0.996667,300,1,0,4.440892e-16
2,0.100326,0.993333,0.993333,299,1,0,2.220446e-16
3,0.154774,0.990000,0.990000,298,1,0,4.440892e-16
4,0.220988,0.986667,0.986667,297,1,0,1.110223e-16
5,0.331674,0.986667,0.986667,296,0,1,1.110223e-16
6,0.369219,0.983322,0.983322,295,1,0,7.771561e-16
7,0.433746,0.979977,0.979977,294,1,0,2.220446e-16
8,0.459292,0.979977,0.979977,293,0,1,2.220446e-16
9,0.469632,0.976621,0.976621,292,1,0,1.110223e-16


The event table records the reconstructed number at risk, events, and censorings at each KM step time, together with the discrepancy between target and reconstructed survival.

In [ ]:
# Save Reconstruction datasets
from pathlib import Path

Path("../data/processed").mkdir(parents=True, exist_ok=True)

df_true.to_csv("../data/processed/true_ipd_trial_001.csv", index=False)
ipd_ours.to_csv("../data/processed/reconstructed_ipd_ours_trial_001.csv", index=False)
ipd_resolve.to_csv("../data/processed/reconstructed_ipd_resolve_trial_001.csv", index=False)
event_table_ours.to_csv("../data/processed/reconstruction_event_table_ours_trial_001.csv", index=False)

## Next Step

The reconstructed IPD datasets will be evaluated in the next notebook using survival-based validation metrics and graphical comparisons to the true KM curve.